# 08 · Compare this run with the paper

`reference_results/` holds the result files used to write the paper. This notebook compares every number in
your new `results/` with them.

**What to expect**
* Zero-shot, lab → field, few-shot, fusion, open-set: identical, or within ~0.1–0.5 points (GPU arithmetic is not
  bit-for-bit identical between runs, so features can differ in the last decimals).
* Fine-tuned CNNs: can differ by a point or two (GPU training is not fully deterministic) — this is normal and is
  mentioned in the paper's limitations (single seed).
* Bootstrap CIs and p-values: tiny differences only.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd()
sys.path.insert(0, str(here if (here / "nbtools.py").exists() else here / "notebooks"))
from nbtools import run, show, figure, where, ROOT
where()

In [ ]:
import pandas as pd, numpy as np
REF, NEW = ROOT / "reference_results", ROOT / "results"
rows = []
for ref in sorted(REF.rglob("*.csv")):
    rel = ref.relative_to(REF)
    new = NEW / rel
    if not new.exists():
        rows.append({"file": str(rel), "status": "not produced yet", "max_diff": np.nan, "values_compared": 0}); continue
    a, b = pd.read_csv(ref), pd.read_csv(new)
    isnum = lambda s: pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
    common = [c for c in a.columns if c in b.columns]
    nums = [c for c in common if isnum(a[c]) and isnum(b[c])]
    keys = [c for c in common if not isnum(a[c]) and not isnum(b[c])]
    if not nums:
        same = a.astype(str).equals(b.astype(str)) if a.shape == b.shape else False
        rows.append({"file": str(rel), "status": "identical" if same else "text differs", "max_diff": np.nan,
                     "values_compared": a.size}); continue
    if keys:
        a2 = a.astype({k: str for k in keys}).groupby(keys)[nums].mean()
        b2 = b.astype({k: str for k in keys}).groupby(keys)[nums].mean()
        a2, b2 = a2.align(b2, join="inner")
    else:
        n = min(len(a), len(b)); a2, b2 = a[nums].iloc[:n], b[nums].iloc[:n]
    d = (a2 - b2).abs().to_numpy(dtype=float)
    md = np.nanmax(d) if np.isfinite(d).any() else 0.0
    status = "identical" if md < 1e-6 else "within 0.1" if md <= 0.1 else "within 0.5" if md <= 0.5 else "differs"
    rows.append({"file": str(rel), "status": status, "max_diff": md, "values_compared": int(np.isfinite(d).sum())})
report = pd.DataFrame(rows)
print(report.status.value_counts().to_string()); print()
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(report.sort_values(["status", "file"]))

## Headline numbers side by side
The key numbers of the abstract, from the paper and from your run.

In [ ]:
def get(base, rel, query, col):
    p = base / rel
    if not p.exists(): return np.nan
    df = pd.read_csv(p).query(query)
    return round(float(df[col].mean()), 1) if len(df) else np.nan

checks = [
    ("BioCLIP 2 lab-trained -> PlantDoc", "fewshot/lab_to_field.csv", "model == 'bioclip2'", "field_accuracy"),
    ("CLIP-L/14 lab-trained -> PlantDoc", "fewshot/lab_to_field.csv", "model == 'clip_l14'", "field_accuracy"),
    ("BioCLIP 2 lab accuracy", "fewshot/lab_to_field.csv", "model == 'bioclip2'", "lab_accuracy"),
    ("BioCLIP 2 K=16 linear probe, PlantDoc", "fewshot/fewshot_runs.csv", "model == 'bioclip2' and K == 16 and method == 'linear probe'", "accuracy"),
    ("BioCLIP 2 K=16 v2 fusion, PlantDoc", "fusion/fusion_runs.csv", "dataset == 'plantdoc' and model == 'bioclip2' and K == 16 and method == 'reliability (v2, ours)'", "accuracy"),
]
out = [{"number": n, "paper": get(REF, r, q, c), "this run": get(NEW, r, q, c)} for n, r, q, c in checks]
out = pd.DataFrame(out); out["difference"] = (out["this run"] - out["paper"]).round(1)
display(out)
print("If a row shows NaN, the method name in that file may differ - check the table above instead.")

## Which way of running is better?
Both run **exactly the same code** (the notebooks call the scripts in `src/`), so the results should agree.
* **Scripts (`run_all.bat`)** — best for the long unattended run; nothing is lost if VS Code closes.
* **Notebooks** — best for understanding and exploring: you see each result right after its step, and you can try
  your own ideas in a new cell (like the *Try it yourself* cell in notebook 03) without touching the paper's code.